This notebook creates the reusable gold-layer base BI view `adwm_wh.gold.vw_bi_factsales_base` for downstream reporting.

Scope:

* Create or replace the base BI view in `adwm_wh.gold`
* Join the gold sales fact to the gold date, customer, product, and employee dimensions
* Expose BI-friendly descriptive fields and reusable measures
* Run lightweight validation for row coverage and unresolved dimension buckets

Business grain:

* One row per sales order line from `adwm_wh.gold.factsales`

Note:

* If some dimensions still resolve to `Unknown`, view creation should continue and validation will surface that condition.

In [0]:
%sql
CREATE OR REPLACE VIEW adwm_wh.gold.vw_bi_factsales_base AS
SELECT
  f.OrderDateKey,
  f.ShipDateKey,
  f.CustomerKey,
  f.EmployeeKey,
  f.ProductKey,
  f.SalesOrderID,
  f.SalesOrderDetailID,
  f.SalesOrderNumber,
  f.SalesOrderLineNumber,
  d.FullDate,
  d.YearNumber,
  d.MonthNumber,
  d.MonthName,
  d.QuarterNumber,
  COALESCE(c.CustomerType, 'Unknown') AS CustomerType,
  COALESCE(c.FullName, 'Unknown') AS CustomerFullName,
  COALESCE(c.AccountNumber, 'Unknown') AS CustomerAccountNumber,
  COALESCE(c.TerritoryName, 'Unknown') AS TerritoryName,
  COALESCE(c.CountryRegion, 'Unknown') AS CountryRegion,
  COALESCE(p.ProductNumber, 'Unknown') AS ProductNumber,
  COALESCE(p.ProductName, 'Unknown') AS ProductName,
  COALESCE(p.CategoryName, 'Unknown') AS CategoryName,
  COALESCE(p.SubcategoryName, 'Unknown') AS SubcategoryName,
  COALESCE(p.Color, 'Unknown') AS Color,
  COALESCE(p.Size, 'Unknown') AS Size,
  COALESCE(e.FullName, 'Unknown') AS EmployeeFullName,
  COALESCE(e.DepartmentName, 'Unknown') AS DepartmentName,
  COALESCE(e.DepartmentGroup, 'Unknown') AS DepartmentGroup,
  COALESCE(e.JobTitle, 'Unknown') AS JobTitle,
  f.OrderQuantity,
  CAST(f.UnitPrice AS DECIMAL(19,4)) AS UnitPrice,
  CAST(f.UnitCost AS DECIMAL(19,4)) AS UnitCost,
  CAST(f.DiscountAmount AS DECIMAL(19,4)) AS DiscountAmount,
  CAST(f.SalesAmount AS DECIMAL(19,4)) AS SalesAmount,
  CAST(f.TotalCost AS DECIMAL(19,4)) AS TotalCost,
  CAST(f.SalesAmount - f.TotalCost AS DECIMAL(19,4)) AS GrossMargin,
  f.FactLoadedAt
FROM adwm_wh.gold.factsales AS f
LEFT JOIN adwm_wh.gold.dimdate AS d
  ON f.OrderDateKey = d.DateKey
LEFT JOIN adwm_wh.gold.dimcustomer AS c
  ON f.CustomerKey = c.CustomerKey
LEFT JOIN adwm_wh.gold.dimproduct AS p
  ON f.ProductKey = p.ProductKey
LEFT JOIN adwm_wh.gold.dimemployee AS e
  ON f.EmployeeKey = e.EmployeeKey;

In [0]:
%sql
SELECT
  COUNT(*) AS row_count,
  MIN(FullDate) AS min_full_date,
  MAX(FullDate) AS max_full_date,
  CAST(SUM(SalesAmount) AS DECIMAL(19,4)) AS total_sales_amount,
  SUM(CASE WHEN TerritoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_territory_rows,
  SUM(CASE WHEN CountryRegion = 'Unknown' THEN 1 ELSE 0 END) AS unknown_country_rows,
  SUM(CASE WHEN CategoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_category_rows,
  SUM(CASE WHEN SubcategoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_subcategory_rows,
  SUM(CASE WHEN EmployeeFullName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_employee_rows
FROM adwm_wh.gold.vw_bi_factsales_base;